# Stage 3: CCIP Embedding Extraction and Mathematical Normalization

## Overview and Purpose
Feature representation is the core foundation of unsupervised visual clustering. Standard pre-trained models (e.g. ImageNet ResNets or general CLIP models) encode general semantic categories (such as animal species, clothing types, or scene environments), rather than visual character identity across diverse poses, lighting, and expressions.

This notebook executes **Stage 3** of the pipeline using **CCIP** (Character-Centered Image Pairing):
1. **CCIP Metric Learning**: Leverages a CAFormer-based visual encoder trained specifically to measure whether two anime image crops depict the same character identity.
2. **Crop Validation**: Pre-validates crop files before inference to ensure corrupt or missing files do not interrupt batch processing.
3. **Model Warmup & Batch Processing**: Warms up inference graphs and batches extraction with memory garbage collection (`gc.collect()`).
4. **Mathematical $L_2$ Normalization**: Normalizes all feature vectors to the unit hypersphere ($L_2$ norm = 1.0).
5. **Persistence**: Saves raw and normalized embedding matrices as `.npy` arrays, a compressed `.npz` bundle, and comprehensive manifest mapping tables.

### Why $L_2$ Normalization?
For normalized unit vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^D$ where $\|\mathbf{u}\|_2 = \|\mathbf{v}\|_2 = 1$:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = \|\mathbf{u}\|_2^2 + \|\mathbf{v}\|_2^2 - 2(\mathbf{u} \cdot \mathbf{v}) = 2 - 2\cos(\theta)$$
Therefore, Euclidean distance in the normalized embedding space directly reflects cosine dissimilarity, allowing standard metric clustering algorithms (e.g. HDBSCAN with Euclidean distance) to cluster on angular similarity.

## 1. Imports and Environment Setup
Import numerical routines and embedding modules.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display

from anime_character_organizer import (
    EmbeddingConfig,
    run_embedding_extraction,
)
from anime_character_organizer.embeddings.normalization import l2_normalize_matrix
from anime_character_organizer.utils.runs import find_latest_run

## 2. Configuration & Model Selection
Configure CCIP model weights, image resolution, and batch sizing.

In [ ]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()
PREVIOUS_CROP_RUN = None  # Auto-discovers latest 02_crop_preparation run

# Pre-trained CCIP model family (options: 'ccip-caformer-24-randaug-pruned', 'ccip-caformer-4')
CCIP_MODEL = "ccip-caformer-24-randaug-pruned"
CCIP_IMAGE_SIZE = 384
BATCH_SIZE = 16
NORMALIZE_EMBEDDINGS = True

print("Embedding Extraction Configuration:")
print(f"  CCIP Model:        {CCIP_MODEL}")
print(f"  Input Image Size:  {CCIP_IMAGE_SIZE}x{CCIP_IMAGE_SIZE}")
print(f"  Batch Size:        {BATCH_SIZE}")
print(f"  L2 Normalization:  {NORMALIZE_EMBEDDINGS}")

## 3. Embedding Extraction Workflow Execution
Extract CCIP features across all crop files. Single-image fallback is automatically employed if a batch encounters unexpected decoding errors.

In [ ]:
results = run_embedding_extraction(
    project_dir=PROJECT_DIR,
    previous_run_dir=PREVIOUS_CROP_RUN,
    ccip_model=CCIP_MODEL,
    ccip_image_size=CCIP_IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    show_progress=True,
)

print(f"Embedding extraction completed.")
print(f"Run directory: {results['run_dir']}")

## 4. Embedding Diagnostics & Mathematical Invariant Verification
Verify matrix dimensions, data types, and check that all rows satisfy the $L_2$ unit norm invariant ($1.0 \pm 10^{-6}$).

In [ ]:
raw_mat = results["raw_embeddings"]
norm_mat = results["normalized_embeddings"]

norms = np.linalg.norm(norm_mat, axis=1)
norm_min = float(norms.min())
norm_max = float(norms.max())

print("Matrix Shapes and Properties:")
print(f"  Raw Embeddings Matrix:        {raw_mat.shape} (dtype: {raw_mat.dtype})")
print(f"  Normalized Embeddings Matrix: {norm_mat.shape} (dtype: {norm_mat.dtype})")
print(f"  L2 Norm Range (Min to Max):   {norm_min:.6f} to {norm_max:.6f}")

assert abs(norm_min - 1.0) < 1e-4, "L2 normalization invariant violated (min < 1.0)"
assert abs(norm_max - 1.0) < 1e-4, "L2 normalization invariant violated (max > 1.0)"
print("Success: Unit norm mathematical invariant verified across all embedding vectors.")

## 5. Successful Embedding Manifest Inspection
The embedding manifest links each matrix row index (`embedding_row`) directly back to its source image path and crop metadata.

In [ ]:
manifest_df = results["successful_manifest_df"]
display(manifest_df[[
    "embedding_row",
    "crop_path",
    "source_path",
    "relative_path",
    "selected_region_type",
    "needs_review",
]].head(25))

## 6. Assumptions, Limitations, and Next Steps

### Assumptions & Limitations
- **Model Lineage**: CCIP models are trained with contrastive loss on anime character pairs. They are sensitive to character identity features (hair, eyes, face structure), but may occasionally confuse close character archetypes (e.g. palette swaps or identical twins).
- **GPU Acceleration**: If CUDA/MPS acceleration is available, `onnxruntime` will automatically leverage it; otherwise inference safely runs on CPU.

### Next Steps
The normalized embedding matrix `ccip_embeddings_l2.npy` and mapping manifest are passed to **Stage 4 (`04_hdbscan_clustering.ipynb`)** for unsupervised character identity clustering.